# exp_4a — Experiment 4A: Transfer Learning with MobileNetV2 and VGG16

Source-based notebook converted from the uploaded experiment code document.


In [ ]:
import zipfile
import os
import shutil

In [ ]:
zip_file = r"D:\DL DATASET\dogs_vs_cats_subset (1) (1).zip"

extract_path = r"D:\DL DATASET\dogs_vs_cats_subset"

In [ ]:
with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2, VGG16
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, Conv2D, MaxPooling2D, Flatten
from tensorflow.keras.models import Sequential, Model
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

IMG_SIZE=(224,224)
BATCH_SIZE=32
EPOCHS=5

In [ ]:
train_dir = r"D:\DL DATASET\dogs_vs_cats_subset\dogs_vs_cats_subset\train"

val_dir = r"D:\DL DATASET\dogs_vs_cats_subset\dogs_vs_cats_subset\val"

In [ ]:
train_gen=ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True)

val_gen=ImageDataGenerator(rescale=1./255)

train_ds=train_gen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary")

val_ds=val_gen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False)

In [ ]:
base=MobileNetV2(weights="imagenet",
                  include_top=False,
                  input_shape=(224,224,3))

base.trainable=False

x=GlobalAveragePooling2D()(base.output)
x=Dense(128,activation="relu")(x)
x=Dropout(0.5)(x)
output=Dense(1,activation="sigmoid")(x)

mobilenet=Model(base.input,output)

mobilenet.compile(optimizer="adam",
                  loss="binary_crossentropy",
                  metrics=["accuracy"])

mobilenet.fit(train_ds,
              validation_data=val_ds,
              epochs=EPOCHS)

# Fine-tuning
base.trainable=True
for layer in base.layers[:-30]:
    layer.trainable=False

mobilenet.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"])

mobilenet.fit(train_ds,
              validation_data=val_ds,
              epochs=2)

In [ ]:
base2=VGG16(weights="imagenet",
             include_top=False,
             input_shape=(224,224,3))

base2.trainable=False

y = GlobalAveragePooling2D()(base2.output)
y=Dense(128,activation="relu")(y)
y=Dropout(0.5)(y)
output2=Dense(1,activation="sigmoid")(y)

vgg=Model(base2.input,output2)

vgg.compile(optimizer="adam",
            loss="binary_crossentropy",
            metrics=["accuracy"])

vgg.fit(train_ds,
        validation_data=val_ds,
        epochs=EPOCHS)